# 12 — Deep Agents: Harness, Planning, Filesystem, Skills, Memory & Subagents

## Learning requirements
Deep Agents là agent harness nằm trên LangChain/LangGraph, cung cấp sẵn các patterns cho long-running agents.

Hiểu:
- `create_deep_agent`;
- planning/todos;
- filesystem/context offloading;
- skills;
- memory;
- custom subagents;
- async subagents (advanced/preview — API có thể thay đổi).

Học Deep Agents sau LangChain/LangGraph để biết harness đang làm hộ phần nào.

In [ ]:
from deepagents import create_deep_agent
from pathlib import Path
import sys, os
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

# Deep Agents accepts provider:model identifiers.
# Keep model configurable instead of relying on a hard-coded latest model.
provider = os.getenv("LLM_PROVIDER", "gemini")
if provider == "gemini":
    deep_model = "google_genai:" + os.getenv("GEMINI_MODEL", "gemini-3.7-flash")
else:
    # For other providers, pass a LangChain model object from src.providers.
    from src.providers import get_chat_model
    deep_model = get_chat_model(provider)

deep_agent = create_deep_agent(
    model=deep_model,
    system_prompt=(
        "You are a research agent. Plan multi-step tasks, keep evidence organized, "
        "and state uncertainty explicitly."
    ),
)

## Skills

Skill = reusable workflow/instruction/resources loaded when relevant.

Suggested structure:

```text
skills/
|- research/SKILL.md
|- code-review/SKILL.md
`- report-writing/SKILL.md
```

Tool vs Skill vs Memory:
- **Tool**: executable capability.
- **Skill**: reusable procedural knowledge/workflow.
- **Memory**: persisted knowledge/context across conversations.

In [ ]:
# Custom subagent definition pattern.
research_subagent = {
    "name": "researcher",
    "description": "Research a bounded technical topic and return evidence + uncertainties.",
    "system_prompt": "You are a specialist researcher. Keep your context focused.",
    "tools": [],
    # "skills": ["/skills/research/"],  # enable after mounting/populating a backend
}

agent_with_subagent = create_deep_agent(
    model=deep_model,
    subagents=[research_subagent],
)

## Memory

Deep Agents memory is filesystem-backed through configured backends.

Design separately:
- agent-scoped memory;
- user-scoped memory;
- organization policy memory;
- read-only vs writable paths.

Shared writable memory is a security risk if untrusted content can modify agent behavior.

## Required output

Build `deep_research_agent` that:
1. plans a 5+ step task;
2. delegates at least one bounded task;
3. offloads intermediate artifacts to files/state;
4. uses 3 skills;
5. documents memory scopes;
6. evaluates whether async subagents materially improve the use case.

## Done criteria
Bạn có thể giải thích khi nào dùng:
`Deep Agents -> LangChain -> LangGraph`
theo mức abstraction/control, chứ không coi chúng là ba framework cạnh tranh.